In [6]:
table_name = "green_202201_202301"

StatementMeta(, bea55170-1e34-4374-8a28-acdf9433e688, 177, Finished, Available, Finished)

In [7]:
df = spark.read.table(table_name)
display(df.limit(10))

StatementMeta(, bea55170-1e34-4374-8a28-acdf9433e688, 178, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5062b430-410e-42ef-a5da-66f96f118ab7)

In [13]:
# Code generated by Data Wrangler for PySpark DataFrame

from pyspark.sql import types as T

def clean_data(df):
    # Select columns: 'VendorID', 'lpep_pickup_datetime' and 7 other columns
    df = df.select('VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'fare_amount', 'total_amount', 'payment_type', 'trip_type')
    # Change column type to int32 for columns: 'VendorID', 'passenger_count' and 2 other columns
    df = df.withColumn('VendorID', df['VendorID'].cast(T.IntegerType()))
    df = df.withColumn('passenger_count', df['passenger_count'].cast(T.IntegerType()))
    df = df.withColumn('payment_type', df['payment_type'].cast(T.IntegerType()))
    df = df.withColumn('trip_type', df['trip_type'].cast(T.IntegerType()))
    # Change column type to float32 for columns: 'trip_distance', 'fare_amount', 'total_amount'
    df = df.withColumn('trip_distance', df['trip_distance'].cast(T.FloatType()))
    df = df.withColumn('fare_amount', df['fare_amount'].cast(T.FloatType()))
    df = df.withColumn('total_amount', df['total_amount'].cast(T.FloatType()))
    # Change column type to datetime64[ns] for columns: 'lpep_pickup_datetime', 'lpep_dropoff_datetime'
    df = df.withColumn('lpep_pickup_datetime', df['lpep_pickup_datetime'].cast(T.TimestampType()))
    df = df.withColumn('lpep_dropoff_datetime', df['lpep_dropoff_datetime'].cast(T.TimestampType()))
    # Replace missing values with -1 in columns: 'VendorID', 'payment_type', 'trip_type'
    df = df.fillna(value=-1, subset=['VendorID', 'payment_type', 'trip_type'])
    # Filter rows based on columns: 'trip_distance', 'fare_amount', 'total_amount'
    df = df.filter((df['trip_distance'] > 0) | (df['fare_amount'] > 0) | (df['total_amount'] > 0) | (df['passenger_count'] > 0))
    return df

df_clean = clean_data(df)
display(df_clean)
print(f"Number of rows = {df_clean.count()}")

StatementMeta(, bea55170-1e34-4374-8a28-acdf9433e688, 184, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 28cb537f-af7a-43c1-b17a-935e75ddbd77)

Number of rows = 908586


In [14]:
from pyspark.sql.functions import date_format

# After casting lpep_pickup_datetime to TimestampType, add:
df_clean = df_clean.withColumn('date_key', date_format('lpep_pickup_datetime', 'yyyyMMdd').cast(T.IntegerType()))
display(df_clean.limit(10))

StatementMeta(, bea55170-1e34-4374-8a28-acdf9433e688, 185, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 83bc9f31-e138-42db-9e64-6b6be6ae667b)

In [ ]:
# Save the results to a new delta table
df_clean.write.format("delta").mode("overwrite").saveAsTable(f"silvercleansed.{table_name}_cleansed")